# Global Model — Held-out Test-Set Evaluation

Scores **one checkpoint** — the final global model in `weights/best_model_global.pt` — on the fixed 80/10/10 held-out test split, and records the **SHA-256 of the weights** alongside every number.

**Why a separate notebook.** In `hto_correction_angles.ipynb` the test evaluation runs right after training, against whatever checkpoint that training run last wrote to the shared path. The global model was retrained on 22 June, so the test numbers in the manuscript (0.48° mean, 1.06° max) and the ones now in the notebook (0.44° mean, 1.34° max) come from two different models. This notebook evaluates a named, hashed file with no training, so the test result is tied to exactly one set of weights — the same weights the OAI validation uses.

**Identical to the training notebook:** the dataset class and deterministic seeded split, letterbox preprocessing, heatmap decoding, Miniaci correction-angle geometry, and the agreement battery (`angle_agreement_report`). The test split is therefore the same 6 radiographs / 12 limb hemispheres.

**Setup.** The container sees `weights/` only if `docker-compose.yml` mounts it: `- ./weights:/tf/weights`.

## Imports & Configuration

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # required by deterministic cuBLAS; must precede CUDA init
import sys, json, math, random, hashlib, datetime
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

for _ckd in ("CKD", "/tf/notebooks/CKD", os.path.join(os.getcwd(), "CKD")):
    if os.path.isdir(_ckd):
        sys.path.append(os.path.abspath(_ckd)); break
else:
    sys.path.append(os.path.abspath("CKD"))
from models import (
    Conformer_tiny_patch16_keypoint_half_heatmap,
    Conformer_small_patch16_keypoint_half_heatmap,
    Conformer_small_patch32_keypoint_half_heatmap,
    Conformer_base_patch16_keypoint_half_heatmap,
)
from utils import extract_coordinates

# ---- must match hto_correction_angles.ipynb -------------------------------------------
DATA_DIR       = "/tf/data/hto/xrays"
COCO_JSON_PATH = os.path.join(DATA_DIR, "hto_annotations.json")
if not os.path.exists(COCO_JSON_PATH):
    COCO_JSON_PATH = "hto_annotations.json"
SEED           = 42
TARGET_SIZE    = 768
HEATMAP_SCALE  = 0.5
SIGMA          = 6.0
SPLIT_RATIOS   = (0.8, 0.1, 0.1)
MODEL_VARIANT  = "small_p16"
CLINICAL_TOLERANCE_DEG = 1.63

# ---- checkpoint under test ----------------------------------------------------------------
WEIGHTS_DIR_CANDIDATES = ["/tf/weights", os.path.join("..", "weights"), "weights"]
CHECKPOINT_FILE        = "best_model_global.pt"
EXPECTED_SHA256        = "20cb99afb0f7455d3c2a71494b586930439b761bbb23482d1d05a977955186d0"   # recorded 2026-09-19

# Other places a global checkpoint has lived; each one found is hashed and compared, so you
# can see whether it is the same file as the one evaluated here.
OTHER_COPIES = ["/tf/notebooks/kfolds_models/best_model_global.pt",
                "/tf/notebooks/best_model_global.pt",
                os.path.join("..", "kfolds_models", "best_model_global.pt")]

OUT_DIR = "test_eval_global"

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUT_DIR, exist_ok=True)

WEIGHTS_DIR = next((d for d in WEIGHTS_DIR_CANDIDATES if os.path.isdir(d)), None)
assert WEIGHTS_DIR is not None, (
    "weights/ not found. Add '- ./weights:/tf/weights' under volumes: in docker-compose.yml "
    "and restart the container.")
CHECKPOINT_PATH = os.path.join(WEIGHTS_DIR, CHECKPOINT_FILE)
assert os.path.isfile(CHECKPOINT_PATH), f"{CHECKPOINT_PATH} not found"
print(f"torch {torch.__version__} | device {device}"
      + (f" | {torch.cuda.get_device_name(0)}" if device.type == "cuda" else ""))
print(f"checkpoint: {os.path.abspath(CHECKPOINT_PATH)}")

## Model Hash
SHA-256 of the checkpoint under test, checked against the recorded value. Any other global-checkpoint copies that exist are hashed too.

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

st = os.stat(CHECKPOINT_PATH)
MODEL_SHA256 = sha256_file(CHECKPOINT_PATH)
HASH_OK = MODEL_SHA256 == EXPECTED_SHA256
manifest = dict(file=CHECKPOINT_FILE, path=os.path.abspath(CHECKPOINT_PATH), bytes=st.st_size,
                modified=datetime.datetime.fromtimestamp(st.st_mtime).isoformat(timespec="seconds"),
                sha256=MODEL_SHA256, expected_sha256=EXPECTED_SHA256, matches_expected=HASH_OK)
for k, v in manifest.items():
    print(f"{k:16s}: {v}")
print("\n" + ("OK: checkpoint matches the recorded SHA-256." if HASH_OK else
              "WARNING: SHA-256 differs from the recorded value — this is not the recorded global model."))

others = []
for p in OTHER_COPIES:
    if os.path.isfile(p):
        h = sha256_file(p)
        others.append(dict(path=os.path.abspath(p), sha256=h, same_as_evaluated=(h == MODEL_SHA256)))
if others:
    print("\nOther global-checkpoint copies found:")
    for o in others:
        print(f"  {'SAME' if o['same_as_evaluated'] else 'DIFFERENT':9s} {o['sha256'][:16]}…  {o['path']}")
manifest["other_copies"] = others

## Preprocessing & Dataset
Copied from `hto_correction_angles.ipynb`. Augmentation is applied only to the train split, so it is not needed here.

In [ ]:
def preprocess_global_image(img, target_size=512):
    """Letterbox-resize *img* to a square canvas of *target_size* pixels."""
    orig_w, orig_h = img.size
    scale   = min(target_size / orig_w, target_size / orig_h)
    new_w   = int(orig_w * scale)
    new_h   = int(orig_h * scale)
    resized = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    pad_left = (target_size - new_w) // 2
    pad_top  = (target_size - new_h) // 2
    final_img = Image.new("RGB", (target_size, target_size), (0, 0, 0))
    final_img.paste(resized, (pad_left, pad_top))
    return final_img, scale, (pad_left, pad_top)


def _rep_x(ann):
    """Representative x-coordinate for hemisphere assignment (mean of visible keypoint x)."""
    kps = ann.get("keypoints", [])
    xs  = [kps[i] for i in range(0, len(kps), 3) if kps[i] > 0]
    if xs:
        return sum(xs) / len(xs)
    bbox = ann.get("bbox", [0, 0, 0, 0])
    return bbox[0] + bbox[2] / 2


class GlobalRadiographKeypointDataset(Dataset):
    """COCO-style dataset for global 12-keypoint detection (verbatim from the training notebook)."""

    def __init__(self, coco_json_path, split="train", split_ratios=(0.8, 0.1, 0.1),
                 target_size=512, heatmap_scale=0.25, sigma=2.0, seed=42, indices=None):
        super().__init__()
        self.target_size   = target_size
        self.heatmap_scale = heatmap_scale
        self.sigma         = sigma
        self.num_keypoints = 12
        self.split         = split

        with open(coco_json_path, "r") as f:
            coco_data = json.load(f)
        images_info = {img["id"]: img for img in coco_data.get("images", [])}
        anns_by_img = {}
        for ann in coco_data.get("annotations", []):
            anns_by_img.setdefault(ann.get("image_id"), []).append(ann)

        valid_samples = []
        for img_id, anns in anns_by_img.items():
            if img_id not in images_info:
                continue
            img_info = images_info[img_id]
            img_w    = img_info.get("width", 2860)
            by_cat = {}
            for ann in anns:
                by_cat.setdefault(ann.get("category_id"), []).append(ann)

            kps_flat = [-1.0, -1.0, 0] * 12
            has_kp   = False
            for cat_id, cat_anns in by_cat.items():
                sorted_anns = sorted(cat_anns, key=_rep_x)
                if len(sorted_anns) == 2:
                    assignments = [(sorted_anns[0], 0), (sorted_anns[1], 6)]
                elif len(sorted_anns) == 1:
                    x    = _rep_x(sorted_anns[0])
                    base = 0 if x < img_w / 2.0 else 6
                    assignments = [(sorted_anns[0], base)]
                else:
                    assignments = [(sorted_anns[0], 0), (sorted_anns[-1], 6)]

                for ann, base in assignments:
                    kps = ann.get("keypoints", [])
                    if cat_id == 1 and len(kps) >= 3:
                        kps_flat[base * 3:(base + 1) * 3] = [kps[0], kps[1], 2 if kps[0] > 0 else 0]
                        if kps[0] > 0:
                            has_kp = True
                    elif cat_id == 2 and len(kps) >= 9:
                        for k in range(3):
                            s = base + 1 + k
                            kps_flat[s * 3:(s + 1) * 3] = [kps[k * 3], kps[k * 3 + 1], kps[k * 3 + 2]]
                            if kps[k * 3 + 2] > 0:
                                has_kp = True
                    elif cat_id == 3 and len(kps) >= 6:
                        for k in range(2):
                            s = base + 4 + k
                            kps_flat[s * 3:(s + 1) * 3] = [kps[k * 3], kps[k * 3 + 1], kps[k * 3 + 2]]
                            if kps[k * 3 + 2] > 0:
                                has_kp = True

            if has_kp:
                filename = img_info.get("file_name")
                img_dir  = os.path.dirname(coco_json_path) or "."
                if not os.path.exists(os.path.join(img_dir, filename)):
                    alt = os.path.join("/tf/data/hto/xrays", os.path.basename(filename))
                    if os.path.exists(alt):
                        img_dir = "/tf/data/hto/xrays"
                valid_samples.append({
                    "img_path":  os.path.join(img_dir, filename),
                    "orig_size": (img_w, img_info.get("height", 8000)),
                    "keypoints": kps_flat,
                })

        # deterministic shuffle, then split
        valid_samples.sort(key=lambda x: x["img_path"])
        random.seed(seed)
        random.shuffle(valid_samples)
        if indices is not None:
            self.samples = [valid_samples[i] for i in indices]
        else:
            n         = len(valid_samples)
            train_end = int(n * split_ratios[0])
            val_end   = train_end + int(n * split_ratios[1])
            self.samples = {"train": valid_samples[:train_end],
                            "val":   valid_samples[train_end:val_end],
                            "test":  valid_samples[val_end:]}.get(split, valid_samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        try:
            img = Image.open(sample["img_path"]).convert("RGB")
        except Exception:
            img = Image.new("RGB", sample["orig_size"], color=(128, 128, 128))
        processed_img, scale, padding = preprocess_global_image(img, self.target_size)

        final_kps, kp_visibility = [], []
        for i in range(self.num_keypoints):
            kp_x = sample["keypoints"][i * 3]; kp_y = sample["keypoints"][i * 3 + 1]; kp_v = sample["keypoints"][i * 3 + 2]
            if kp_v > 0 and kp_x >= 0 and kp_y >= 0:
                final_kps.append([kp_x * scale + padding[0], kp_y * scale + padding[1]]); kp_visibility.append(1.0)
            else:
                final_kps.append([-1.0, -1.0]); kp_visibility.append(0.0)

        img_tensor = torch.from_numpy(np.array(processed_img)).permute(2, 0, 1).float() / 255.0
        img_tensor = (img_tensor - torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)) \
                     / torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        return {"image": img_tensor,
                "keypoint": torch.tensor(final_kps, dtype=torch.float32),
                "visibility": torch.tensor(kp_visibility, dtype=torch.float32),
                "img_path": sample["img_path"],
                "orig_size": torch.tensor(sample["orig_size"])}

## Test Split
The split is reproduced from the seed; its file list is printed and fingerprinted so the evaluated set can be cited and checked.

In [ ]:
assert os.path.exists(COCO_JSON_PATH), f"annotation file not found: {COCO_JSON_PATH}"
test_ds = GlobalRadiographKeypointDataset(COCO_JSON_PATH, split="test", split_ratios=SPLIT_RATIOS,
                                          target_size=TARGET_SIZE, heatmap_scale=HEATMAP_SCALE,
                                          sigma=SIGMA, seed=SEED)
_all = GlobalRadiographKeypointDataset(COCO_JSON_PATH, split="all", target_size=TARGET_SIZE, seed=SEED)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=0)

TEST_FILES = [os.path.basename(s["img_path"]) for s in test_ds.samples]
SPLIT_SHA256 = hashlib.sha256("\n".join(sorted(TEST_FILES)).encode()).hexdigest()
print(f"radiographs: {len(_all)} total | test split: {len(test_ds)}")
for f in TEST_FILES:
    print("  ", f)
print(f"test-split fingerprint (sha256 of sorted file names): {SPLIT_SHA256[:16]}…")

## Load the Checkpoint

In [ ]:
_model_map = {
    "tiny":      Conformer_tiny_patch16_keypoint_half_heatmap,
    "small_p16": Conformer_small_patch16_keypoint_half_heatmap,
    "small_p32": Conformer_small_patch32_keypoint_half_heatmap,
    "base":      Conformer_base_patch16_keypoint_half_heatmap,
}
model_global = _model_map[MODEL_VARIANT](num_keypoints=12).to(device)
model_global.load_state_dict(torch.load(CHECKPOINT_PATH, weights_only=True, map_location=device))
model_global.eval()
n_params = sum(p.numel() for p in model_global.parameters())
print(f"Loaded {CHECKPOINT_FILE} ({MODEL_VARIANT}, {n_params/1e6:.2f} M params) | sha256 {MODEL_SHA256}")

## Correction-Angle Geometry & Agreement Statistics
Copied from `hto_correction_angles.ipynb`.

In [ ]:
def map_global_to_orig(kp_final, orig_size, target_size=512):
    """Invert the letterbox transform back to original image coordinates."""
    orig_w, orig_h = orig_size
    scale    = min(target_size / orig_w, target_size / orig_h)
    pad_left = (target_size - int(orig_w * scale)) // 2
    pad_top  = (target_size - int(orig_h * scale)) // 2
    return np.array([(kp_final[0] - pad_left) / scale, (kp_final[1] - pad_top) / scale])


def calculate_intersection(p1, p2, target_y):
    """X-coordinate where line (p1 -> p2) crosses y = *target_y*."""
    if p2[0] == p1[0]: return p1[0]
    m = (p2[1] - p1[1]) / (p2[0] - p1[0])
    if m == 0:
        return p1[0] if abs(target_y - p1[1]) < 1e-9 else float("nan")
    return (target_y - p1[1]) / m + p1[0]


def evaluate_side_geometry(points):
    """Miniaci correction angle for one hemisphere -> (alpha, fujisawa, ankle_c, target_at_ankle)."""
    ankle_c  = (points["ankle_inner"] + points["ankle_outer"]) / 2.0
    fujisawa = points["knee_inner"] + 0.625 * (points["knee_outer"] - points["knee_inner"])
    tx              = calculate_intersection(points["femur_head"], fujisawa, ankle_c[1])
    target_at_ankle = np.array([tx, ankle_c[1]])
    v_orig   = ankle_c         - points["ost_point"]
    v_target = target_at_ankle - points["ost_point"]
    raw      = abs(math.atan2(v_orig[1], v_orig[0]) - math.atan2(v_target[1], v_target[0]))
    alpha    = min(raw, 2 * math.pi - raw) * 180.0 / math.pi
    return alpha, fujisawa, ankle_c, target_at_ankle


_SIDE_KEYS = ["femur_head", "knee_inner", "ost_point", "knee_outer", "ankle_inner", "ankle_outer"]


def icc21_manual(ratings):
    ratings = np.asarray(ratings, dtype=float); n, k = ratings.shape; grand = ratings.mean()
    row_means = ratings.mean(axis=1, keepdims=True); col_means = ratings.mean(axis=0, keepdims=True)
    ss_total = ((ratings - grand) ** 2).sum()
    ss_row = k * ((row_means - grand) ** 2).sum(); ss_col = n * ((col_means - grand) ** 2).sum()
    ss_err = ss_total - ss_row - ss_col
    ms_row = ss_row / (n - 1); ms_col = ss_col / (k - 1); ms_err = ss_err / ((n - 1) * (k - 1))
    return float((ms_row - ms_err) / (ms_row + (k - 1) * ms_err + (k / n) * (ms_col - ms_err)))


def compute_icc(gt, pred):
    gt = np.asarray(gt, dtype=float); pred = np.asarray(pred, dtype=float); n = len(gt)
    try:
        import pingouin as pg
        long = pd.DataFrame({"target": list(range(n)) * 2,
                             "rater": ["mean_observer"] * n + ["auto"] * n,
                             "angle": np.concatenate([gt, pred])})
        res = pg.intraclass_corr(data=long, targets="target", raters="rater", ratings="angle")
        row = res.loc[res["Type"].isin(["ICC2", "ICC(A,1)"])].iloc[0]
        ci = row["CI95%" if "CI95%" in res.columns else "CI95"]
        return float(row["ICC"]), float(ci[0]), float(ci[1]), "pingouin ICC(2,1)"
    except ImportError:
        return (icc21_manual(np.column_stack([gt, pred])), np.nan, np.nan,
                "manual ICC(2,1) -- pip install pingouin for the 95% CI")


def angle_agreement_report(gt, pred, tolerance=CLINICAL_TOLERANCE_DEG, label=""):
    gt = np.asarray(gt, dtype=float); pred = np.asarray(pred, dtype=float)
    diff = pred - gt; abs_err = np.abs(diff)
    icc, lo, hi, method = compute_icc(gt, pred)
    bias = diff.mean(); sd_diff = diff.std(ddof=1)
    loa_low, loa_high = bias - 1.96 * sd_diff, bias + 1.96 * sd_diff
    pearson = np.corrcoef(gt, pred)[0, 1] if len(gt) > 1 else float("nan")
    within = 100.0 * np.mean(abs_err <= tolerance); rmse = float(np.sqrt(np.mean(diff ** 2)))
    ci_str = "" if np.isnan(lo) else f" ({lo:.3f}-{hi:.3f} 95% CI)"
    print("=" * 64); print(f"  {label}   (n = {len(gt)} limb hemispheres)"); print("=" * 64)
    print("Absolute correction-angle error (degrees)")
    print(f"  mean    : {abs_err.mean():.4f}");  print(f"  median  : {np.median(abs_err):.4f}")
    print(f"  std     : {abs_err.std(ddof=1):.4f}"); print(f"  min     : {abs_err.min():.4f}")
    print(f"  max     : {abs_err.max():.4f}");   print(f"  RMSE    : {rmse:.4f}")
    print(f"  within +/-{tolerance:.2f} deg : {within:.1f}%")
    print("\nAgreement"); print(f"  ICC(2,1) : {icc:.3f}{ci_str}   [{method}]"); print(f"  Pearson r: {pearson:.4f}")
    print("\nBland-Altman (predicted - GT)")
    print(f"  bias (mean diff)       : {bias:+.4f}")
    print(f"  95% limits of agreement: [{loa_low:+.4f}, {loa_high:+.4f}]")
    print("=" * 64 + "\n")
    return {"label": label, "n": int(len(gt)), "mean": float(abs_err.mean()), "median": float(np.median(abs_err)),
            "std": float(abs_err.std(ddof=1)), "min": float(abs_err.min()), "max": float(abs_err.max()),
            "rmse": rmse, "within_tol_pct": float(within), "icc": float(icc), "icc_lo": float(lo),
            "icc_hi": float(hi), "ba_bias": float(bias), "ba_loa_low": float(loa_low),
            "ba_loa_high": float(loa_high), "pearson": float(pearson)}

## Inference on the Test Split

In [ ]:
rows = []
with torch.no_grad():
    for batch in test_loader:
        imgs = batch["image"].to(device)
        coords_batch = extract_coordinates(torch.sigmoid(model_global(imgs)).cpu(),
                                           scale_factor=1.0 / HEATMAP_SCALE).numpy()
        gts_batch, orig_sizes, paths = batch["keypoint"].numpy(), batch["orig_size"].numpy(), batch["img_path"]
        for b in range(len(imgs)):
            for base, side in [(0, "left"), (6, "right")]:
                pts_gt, pts_pred = {}, {}
                for k_off, name in enumerate(_SIDE_KEYS):
                    slot = base + k_off
                    if gts_batch[b][slot][0] >= 0:
                        pts_gt[name]   = map_global_to_orig(gts_batch[b][slot],    orig_sizes[b], TARGET_SIZE)
                        pts_pred[name] = map_global_to_orig(coords_batch[b][slot], orig_sizes[b], TARGET_SIZE)
                if all(k in pts_gt for k in _SIDE_KEYS) and all(k in pts_pred for k in _SIDE_KEYS):
                    gt_alpha,   *_ = evaluate_side_geometry(pts_gt)
                    pred_alpha, *_ = evaluate_side_geometry(pts_pred)
                    rows.append(dict(file=os.path.basename(paths[b]), image_hemisphere=side,
                                     gt_angle=float(gt_alpha), pred_angle=float(pred_alpha),
                                     error=float(pred_alpha - gt_alpha),
                                     abs_error=float(abs(pred_alpha - gt_alpha)),
                                     within_tolerance=bool(abs(pred_alpha - gt_alpha) <= CLINICAL_TOLERANCE_DEG),
                                     model_sha256=MODEL_SHA256))
T = pd.DataFrame(rows)
print(f"{len(T)} limb hemispheres from {T['file'].nunique()} radiographs")
print(T[["file", "image_hemisphere", "gt_angle", "pred_angle", "error"]].to_string(index=False, float_format=lambda v: f"{v:.3f}"))

## Results

The agreement battery, then a comparison with the two test results already on record. Whichever one this checkpoint reproduces is the model that produced it.

In [ ]:
stats = angle_agreement_report(T["gt_angle"].values, T["pred_angle"].values,
                               label=f"Held-out test split — {CHECKPOINT_FILE} ({MODEL_SHA256[:12]}…)")

KNOWN = {   # test-split results already on record
    "manuscript v1/v2 (pre-22-June model)":                 dict(mean=0.48,   median=0.45,   max=1.06,   icc=0.987),
    "hto_correction_angles.ipynb @710bc10 (22-June model)": dict(mean=0.4361, median=0.3156, max=1.3416, icc=0.988),
}
print(f"{'':52s}{'mean':>8}{'median':>8}{'max':>8}{'ICC':>8}")
print(f"{'this checkpoint':52s}{stats['mean']:8.3f}{stats['median']:8.3f}{stats['max']:8.3f}{stats['icc']:8.3f}")
for name, k in KNOWN.items():
    print(f"{name:52s}{k['mean']:8.3f}{k['median']:8.3f}{k['max']:8.3f}{k['icc']:8.3f}")
matches = [name for name, k in KNOWN.items()
           if abs(stats["mean"] - k["mean"]) < 0.006 and abs(stats["max"] - k["max"]) < 0.006
           and abs(stats["median"] - k["median"]) < 0.006]
print("\n=> " + (f"reproduces: {matches[0]}" if matches else "matches neither result on record"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
lo, hi = min(T.gt_angle.min(), T.pred_angle.min()) - 1, max(T.gt_angle.max(), T.pred_angle.max()) + 1
ax1.scatter(T.gt_angle, T.pred_angle, color="#C44E52", zorder=3)
ax1.plot([lo, hi], [lo, hi], "k--", lw=1)
ax1.set_xlabel("Mean-observer correction angle (°)"); ax1.set_ylabel("Predicted correction angle (°)")
ax1.set_title(f"Test split (n={len(T)})  ICC={stats['icc']:.3f}"); ax1.grid(alpha=0.3)
mean_ = (T.gt_angle + T.pred_angle) / 2
ax2.scatter(mean_, T.error, color="#C44E52", zorder=3)
for y, ls in [(stats["ba_bias"], "-"), (stats["ba_loa_low"], "--"), (stats["ba_loa_high"], "--")]:
    ax2.axhline(y, color="k", ls=ls, lw=1)
for y in (-CLINICAL_TOLERANCE_DEG, CLINICAL_TOLERANCE_DEG):
    ax2.axhline(y, color="#4C72B0", ls=":", lw=1)
ax2.set_xlabel("Mean of predicted & observer (°)"); ax2.set_ylabel("Predicted − observer (°)")
ax2.set_title(f"Bland–Altman (bias {stats['ba_bias']:+.2f}°; dotted = ±{CLINICAL_TOLERANCE_DEG}°)"); ax2.grid(alpha=0.3)
fig.suptitle(f"{CHECKPOINT_FILE}  sha256 {MODEL_SHA256[:16]}…", fontsize=10)
plt.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "test_agreement.png"), dpi=150); plt.show()

## Save

In [ ]:
T.to_csv(os.path.join(OUT_DIR, "test_hemispheres.csv"), index=False)
record = dict(evaluated_at=datetime.datetime.now().isoformat(timespec="seconds"),
              checkpoint=manifest, test_split=dict(files=TEST_FILES, sha256=SPLIT_SHA256, n_radiographs=len(test_ds),
                                                   n_hemispheres=int(len(T)), seed=SEED, split_ratios=list(SPLIT_RATIOS)),
              environment=dict(torch=torch.__version__, device=str(device),
                               gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else None),
              results=stats, reproduces=matches)
with open(os.path.join(OUT_DIR, "test_results.json"), "w") as f:
    json.dump(record, f, indent=2, default=float)
print(f"Saved: {OUT_DIR}/test_hemispheres.csv, {OUT_DIR}/test_results.json, {OUT_DIR}/test_agreement.png")